# Bank Statement PDF → Excel Extraction

Convert **Allied Bank (myABL)** account-statement PDFs into clean, tabular data and export to Excel with a monthly income/expense analysis.

**Pipeline**
1. **Extract** — read every page with `pdfplumber` and reconstruct rows from word coordinates.
2. **Classify** — assign each amount to *Debit*, *Credit* or *Balance* using its column position (the statement's fixed layout), not fragile text parsing.
3. **Structure** — build a tidy `pandas.DataFrame` with parsed dates and numeric amounts.
4. **Validate** — reconcile the running balance against the statement's opening/closing balance.
5. **Export** — write transactions + a monthly summary to an `.xlsx` workbook.

> The extractor is position-based, so multi-line descriptions and the Debit/Credit split are handled reliably.


In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import pdfplumber

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## 1. Configuration

Point `STATEMENTS_DIR` at your PDFs. If a statement is password-protected, add its password in `PASSWORDS` keyed by file name.

The column anchors below are the right-edge x-coordinates of the *Debit*, *Credit* and *Balance* columns, read directly from the statement header. Amounts are right-aligned, so an amount belongs to whichever column its right edge sits closest to.


In [ ]:
# --- Paths -------------------------------------------------------------------
# Resolve the project root so the notebook works no matter the current folder.
def _project_root() -> Path:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / "finlib").is_dir() or (cand / "requirements.txt").is_file():
            return cand
    return Path.cwd()

ROOT = _project_root()
STATEMENTS_DIR = ROOT / "data" / "statements"
OUTPUT_DIR = ROOT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Passwords for protected PDFs, keyed by file name. Leave empty if none.
PASSWORDS: dict[str, str] = {
    # "Account Statement.pdf": "905003",
}

# --- Shared patterns ---------------------------------------------------------
MONEY_RE = re.compile(r"^-?[\d,]+\.\d{2}$")

# --- Per-bank layout definitions ---------------------------------------------
# `anchors` are the right-edge x-coordinates of each amount column. Amounts are
# right-aligned, so a value is assigned to the column its right edge sits nearest.
# `detect` picks the layout from the first page's text.
LAYOUTS: dict[str, dict] = {
    "myabl": {
        "detect": lambda t: "Opening Balance:" in t and "Account Statement" in t,
        "anchors": {"debit": 372.68, "credit": 471.89, "balance": 571.10},
        "date_re": re.compile(r"^\d{2}\s[A-Z][a-z]{2}\s\d{4}$"),
        "date_fmt": "%d %b %Y",
        "date_x_max": 100.0,          # date tokens sit left of this x
        "desc_x0_min": 100.0,
        "desc_x1_max": 340.0,
        "header_bottom": 205.0,       # fallback when the header row isn't found
        "footer_re": re.compile(r"^\d+\s+\d{2}\s\w{3}\s\d{4},"),
        "order": "asc",              # rows printed oldest -> newest
    },
    "hbl": {
        "detect": lambda t: "HBL Mobile" in t or "Account Activity generated" in t,
        "anchors": {"debit": 375.0, "credit": 446.0, "balance": 550.0},
        "date_re": re.compile(r"^\d{2}-\d{2}-\d{4}$"),
        "date_fmt": "%d-%m-%Y",
        "date_x_max": 90.0,           # keeps Date, drops the Value Date column
        "desc_x0_min": 150.0,
        "desc_x1_max": 340.0,
        "header_bottom": 218.0,
        "footer_re": None,
        "order": "desc",             # rows printed newest -> oldest
    },
}

## 2. Core extraction

`parse_statement()` opens a PDF and returns two things:

- **metadata** — account number, title, currency, opening/closing balance (read once from page 1).
- **transactions** — one record per posted transaction, with multi-line descriptions merged.

Each amount is placed in Debit / Credit / Balance purely by its x-position, so we never have to guess whether a value is money-in or money-out.


In [ ]:
@dataclass
class Statement:
    """Parsed contents of one statement PDF."""

    metadata: dict = field(default_factory=dict)
    transactions: list[dict] = field(default_factory=list)


def _detect_layout(page_text: str) -> str:
    """Identify which bank layout a statement uses from its first page."""
    for name, cfg in LAYOUTS.items():
        if cfg["detect"](page_text):
            return name
    raise ValueError("Unrecognised statement layout")


def _classify_amount(x1: float, anchors: dict) -> str:
    """Map an amount's right edge (x1) to Debit / Credit / Balance."""
    debit, credit, balance = anchors["debit"], anchors["credit"], anchors["balance"]
    if x1 < (debit + credit) / 2:
        return "debit"
    if x1 < (credit + balance) / 2:
        return "credit"
    return "balance"


def _group_words_into_lines(words: list[dict], tol: float = 3.0) -> list[list[dict]]:
    """Cluster words that share (approximately) the same vertical position."""
    lines: dict[int, list[dict]] = {}
    for w in words:
        lines.setdefault(round(w["top"] / tol), []).append(w)
    return [sorted(lines[k], key=lambda w: w["x0"]) for k in sorted(lines)]


def _extract_metadata(page, layout_name: str) -> dict:
    """Read the account header block, using layout-specific field patterns."""
    text = page.extract_text() or ""
    meta: dict = {}
    if layout_name == "myabl":
        patterns = {
            "account_number": r"Account Number:\s*(\S+)",
            "account_title": r"Account Title:\s*(.+)",
            "currency": r"Currency:\s*(\S+)",
            "opening_balance": r"Opening Balance:\s*([\d,]+\.\d{2})",
            "closing_balance": r"Closing Balance:\s*([\d,]+\.\d{2})",
        }
        for key, pat in patterns.items():
            m = re.search(pat, text)
            meta[key] = m.group(1).strip() if m else None
    else:  # hbl
        m_title = re.search(r"AccountTitle:(.+)", text)
        m_iban = re.search(r"IBAN:(\S+)", text)
        # e.g. "08747901905003 6110181572381 PKR 0.00 38,274.76"
        m_row = re.search(r"(\d{6,})\s+\d+\s+([A-Z]{3})\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})", text)
        meta["account_number"] = m_row.group(1) if m_row else None
        meta["account_title"] = m_title.group(1).strip() if m_title else None
        meta["currency"] = m_row.group(2) if m_row else None
        meta["opening_balance"] = m_row.group(3) if m_row else None
        meta["closing_balance"] = m_row.group(4) if m_row else None
        meta["iban"] = m_iban.group(1) if m_iban else None
    return meta


def parse_statement(path: str | Path, password: str | None = None) -> Statement:
    """Extract account metadata and transactions from a statement PDF."""
    path = Path(path)
    stmt = Statement()

    with pdfplumber.open(str(path), password=password) as pdf:
        layout_name = _detect_layout(pdf.pages[0].extract_text() or "")
        layout = LAYOUTS[layout_name]
        anchors = layout["anchors"]

        stmt.metadata = _extract_metadata(pdf.pages[0], layout_name)
        stmt.metadata.update(
            source_file=path.name,
            pages=len(pdf.pages),
            layout=layout_name,
            date_fmt=layout["date_fmt"],
            order=layout["order"],
        )

        for page in pdf.pages:
            words = page.extract_words(keep_blank_chars=False)

            # Find the column-header row on this page so repeated headers are skipped.
            header_tops = [w["top"] for w in words if w["text"] == "Description"]
            start_y = max(header_tops) if header_tops else layout["header_bottom"]

            for row in _group_words_into_lines(words):
                if not row or row[0]["top"] <= start_y + 2:
                    continue
                line_text = " ".join(w["text"] for w in row)
                if layout["footer_re"] and layout["footer_re"].match(line_text):
                    continue

                date_str = " ".join(w["text"] for w in row if w["x0"] < layout["date_x_max"])
                desc = " ".join(
                    w["text"]
                    for w in row
                    if layout["desc_x0_min"] <= w["x0"] and w["x1"] <= layout["desc_x1_max"]
                )
                amounts = {
                    _classify_amount(w["x1"], anchors): w["text"]
                    for w in row
                    if MONEY_RE.match(w["text"])
                }

                if layout["date_re"].match(date_str):
                    stmt.transactions.append(
                        {
                            "date": date_str,
                            "description": desc,
                            "debit": amounts.get("debit"),
                            "credit": amounts.get("credit"),
                            "balance": amounts.get("balance"),
                        }
                    )
                elif stmt.transactions and desc:
                    # Continuation line: append to the current transaction's description.
                    stmt.transactions[-1]["description"] += " " + desc

    return stmt


## 3. Build a tidy DataFrame

Convert the raw records into a typed table: real `datetime` dates, numeric `debit`/`credit`/`balance`, a signed `amount` (credit positive, debit negative) and a `month` period for grouping.


In [ ]:
def _to_number(series: pd.Series) -> pd.Series:
    """Convert '1,234.56' style strings to float, empty/None -> 0.0."""
    return (
        series.fillna("")
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace("", "0")
        .astype(float)
    )


def statement_to_dataframe(stmt: Statement) -> pd.DataFrame:
    """Turn a parsed Statement into a typed, analysis-ready DataFrame."""
    records = stmt.transactions
    # Statements printed newest-first are reversed so rows run oldest -> newest.
    if stmt.metadata.get("order") == "desc":
        records = list(reversed(records))

    df = pd.DataFrame(records)
    if df.empty:
        return df

    df["date"] = pd.to_datetime(df["date"], format=stmt.metadata.get("date_fmt"))
    for col in ("debit", "credit", "balance"):
        df[col] = _to_number(df[col])

    df["amount"] = df["credit"] - df["debit"]        # + inflow, - outflow
    df["type"] = df["amount"].apply(lambda v: "Credit" if v >= 0 else "Debit")
    df["month"] = df["date"].dt.to_period("M").astype(str)
    df["source_file"] = stmt.metadata.get("source_file")

    # Keep the statement's posting order (reversed above for newest-first files);
    # it follows the running balance even when rows aren't strictly date-sorted.
    df = df.reset_index(drop=True)
    return df[
        ["date", "month", "description", "type", "debit", "credit", "amount", "balance", "source_file"]
    ]


## 4. Run the extraction

Discover every PDF in the statements folder, parse each one, and combine the results into a single DataFrame.


In [ ]:
statements: list[Statement] = []
frames: list[pd.DataFrame] = []


for pdf_path in sorted(STATEMENTS_DIR.glob("*.pdf")):
    password = PASSWORDS.get(pdf_path.name)
    try:
        stmt = parse_statement(pdf_path, password=password)
    except Exception as exc:  # noqa: BLE001 - report and continue with other files
        print(f"SKIPPED  {pdf_path.name}: {type(exc).__name__}: {exc}")
        continue

    df = statement_to_dataframe(stmt)
    if df.empty:
        print(f"WARNING  {pdf_path.name}: 0 transactions extracted (unrecognised layout?)")
        continue

    statements.append(stmt)
    frames.append(df)
    print(
        f"OK       {pdf_path.name}: {len(df)} transactions across "
        f"{stmt.metadata['pages']} pages [{stmt.metadata['layout']}]"
    )

transactions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal transactions: {len(transactions)}")
transactions.head(10)


## 5. Validate: reconcile the running balance

The strongest correctness check for a bank statement: starting from the **opening balance** and applying every credit and debit must land exactly on the **closing balance** (`closing_ok`). This validates the Debit/Credit classification for the whole statement.

`row_mismatches` additionally checks each printed balance against the running total in posting order. A few mismatches can occur when a bank lists same-day transactions in an order that doesn't follow the running balance — these are ordering quirks in the source PDF, not extraction errors, as long as `closing_ok` is `True`.


In [ ]:
def reconcile(stmt: Statement, df: pd.DataFrame, tol: float = 0.01) -> dict:
    """Replay transactions from the opening balance and compare to printed balances."""
    opening = float((stmt.metadata.get("opening_balance") or "0").replace(",", ""))
    closing = float((stmt.metadata.get("closing_balance") or "0").replace(",", ""))

    running = opening + df["amount"].cumsum()
    net = round(float(df["amount"].sum()), 2)

    return {
        "source_file": stmt.metadata.get("source_file"),
        "layout": stmt.metadata.get("layout"),
        "transactions": len(df),
        "opening_balance": opening,
        "closing_balance": closing,
        "computed_closing": round(float(running.iloc[-1]) if len(running) else opening, 2),
        "row_mismatches": int((running.sub(df["balance"]).abs() > tol).sum()),
        "closing_ok": abs(opening + net - closing) <= tol,
    }


report = (
    pd.DataFrame(reconcile(s, f) for s, f in zip(statements, frames))
    if statements
    else pd.DataFrame()
)
report


## 6. Monthly income / expense analysis

Aggregate transactions by month: total money in (credits), total money out (debits), net change and transaction count.


In [ ]:
def monthly_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Per-month totals of income (credit), expense (debit) and net change."""
    if df.empty:
        return df
    summary = (
        df.groupby("month")
        .agg(
            transactions=("amount", "size"),
            income=("credit", "sum"),
            expense=("debit", "sum"),
            net=("amount", "sum"),
        )
        .reset_index()
    )
    summary["closing_balance"] = df.groupby("month")["balance"].last().values
    return summary.round(2)


monthly = monthly_summary(transactions)
monthly


## 7. Export an interactive Excel dashboard

Build a polished workbook with an interactive **Dashboard**:

- **Period** and **Account** dropdowns (All / year / month) that drive everything.
- **KPI cards** — income, expense, net and transaction count for the current selection (live `SUMIFS`).
- **Pie chart** of income vs expense, plus **bar** and **line** charts of the monthly trend.
- A **live transaction list** (`FILTER`) that shows only the rows matching the selection.

Supporting sheets: **Transactions** (a filterable Excel table), **Monthly Summary**, and **Reconciliation**.

> The dropdown-driven KPIs, charts and transaction list recalculate automatically in Excel. The live list uses `FILTER` (Excel 365 / 2021+); on older Excel, use the AutoFilter on the **Transactions** sheet instead.


In [ ]:
from openpyxl.chart import BarChart, LineChart, PieChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo

# --- palette -----------------------------------------------------------------
NAVY = "1F3864"
BLUE = "2E75B6"
GREEN = "548235"
RED = "C00000"
LIGHT = "D9E1F2"
GREY = "808080"
WHITE = "FFFFFF"

_thin = Side(style="thin", color="BFBFBF")
_BORDER = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
_MONEY_FMT = "#,##0.00"
_INT_FMT = "#,##0"
_DATE_FMT = "yyyy-mm-dd"
_TABLE_STYLE = TableStyleInfo(
    name="TableStyleMedium2", showRowStripes=True, showColumnStripes=False
)


def _acct_labels(transactions: pd.DataFrame, report: pd.DataFrame) -> pd.Series:
    """Friendly account label per row, e.g. 'MYABL (1789...pdf)'."""
    layout_by_file = (
        dict(zip(report["source_file"], report["layout"])) if not report.empty else {}
    )
    def label(sf: str) -> str:
        lay = layout_by_file.get(sf, "")
        return f"{lay.upper()} ({sf})" if lay else str(sf)
    return transactions["source_file"].map(label)


def _style_table_sheet(ws, df, table_name, *, tab_color=NAVY):
    """Give a data sheet the shared look: table, banding, formats, freeze."""
    ws.sheet_view.showGridLines = False
    ws.sheet_properties.tabColor = tab_color
    nrows, ncols = df.shape
    ref = f"A1:{get_column_letter(ncols)}{nrows + 1}"
    table = Table(displayName=table_name, ref=ref)
    table.tableStyleInfo = _TABLE_STYLE
    ws.add_table(table)
    ws.freeze_panes = "A2"
    for idx, col in enumerate(df.columns, start=1):
        letter = get_column_letter(idx)
        if col == "date":
            fmt = _DATE_FMT
        elif pd.api.types.is_float_dtype(df[col].dtype):
            fmt = _MONEY_FMT
        elif pd.api.types.is_integer_dtype(df[col].dtype):
            fmt = _INT_FMT
        else:
            fmt = None
        if fmt:
            for cell in ws[letter][1:]:
                cell.number_format = fmt
        width = max([len(str(col))] + df[col].astype(str).str.len().tolist())
        ws.column_dimensions[letter].width = min(max(width + 2, 10), 55)


def _kpi_card(ws, cell_label, cell_value, title, formula, *, fmt=_MONEY_FMT, color=NAVY):
    """Draw a titled KPI card (label cell above value cell)."""
    lab = ws[cell_label]
    lab.value = title
    lab.font = Font(bold=True, color=WHITE, size=10)
    lab.fill = PatternFill("solid", fgColor=color)
    lab.alignment = Alignment(horizontal="center", vertical="center")
    lab.border = _BORDER
    val = ws[cell_value]
    val.value = formula
    val.font = Font(bold=True, size=14, color=color)
    val.alignment = Alignment(horizontal="center", vertical="center")
    val.number_format = fmt
    val.fill = PatternFill("solid", fgColor=LIGHT)
    val.border = _BORDER


def build_dashboard_workbook(transactions, monthly, report, path):
    """Write a filterable, chart-rich Excel dashboard from the extracted data."""
    path = Path(path)
    if transactions.empty:
        raise ValueError("No transactions to export")

    tx = transactions.copy()
    tx.insert(2, "account", _acct_labels(tx, report))
    n = len(tx)
    last = n + 1  # last data row (row 1 is the header)

    months = sorted(tx["month"].unique().tolist())
    years = sorted({m[:4] for m in months})
    periods = ["All"] + years + months
    accounts = ["All"] + sorted(tx["account"].unique().tolist())

    with pd.ExcelWriter(path, engine="openpyxl", datetime_format=_DATE_FMT) as writer:
        book = writer.book
        # Force Excel to recalculate every formula on open so the KPIs, charts and
        # the filtered transaction list populate immediately (openpyxl caches none).
        book.calculation.fullCalcOnLoad = True

        # --- data sheets (shared styling) ------------------------------------
        tx_out = tx.copy()
        tx_out["date"] = pd.to_datetime(tx_out["date"]).dt.date
        tx_out.to_excel(writer, sheet_name="Transactions", index=False)
        monthly.to_excel(writer, sheet_name="Monthly Summary", index=False)
        report.to_excel(writer, sheet_name="Reconciliation", index=False)

        _style_table_sheet(writer.sheets["Transactions"], tx_out, "tblTxns")
        _style_table_sheet(writer.sheets["Monthly Summary"], monthly, "tblMonthly", tab_color=BLUE)
        _style_table_sheet(writer.sheets["Reconciliation"], report, "tblRecon", tab_color=GREEN)

        # --- hidden list + calc sheets ---------------------------------------
        ws_lists = book.create_sheet("Lists")
        for i, p in enumerate(periods, start=1):
            ws_lists.cell(row=i, column=1, value=p)
        for i, a in enumerate(accounts, start=1):
            ws_lists.cell(row=i, column=3, value=a)
        ws_lists.sheet_state = "hidden"

        ws_calc = book.create_sheet("Calc")
        # Selection -> SUMIFS criteria (wildcard handles All / year / month).
        ws_calc["B1"] = '=IF(Dashboard!$C$4="All","*",Dashboard!$C$4&"*")'
        ws_calc["B2"] = '=IF(Dashboard!$C$5="All","*",Dashboard!$C$5)'
        # Pie source (income vs expense for the current selection).
        ws_calc["D1"], ws_calc["E1"] = "Category", "Amount"
        ws_calc["D2"], ws_calc["E2"] = "Income", (
            "=SUMIFS(Transactions!$G:$G,Transactions!$B:$B,$B$1,Transactions!$C:$C,$B$2)"
        )
        ws_calc["D3"], ws_calc["E3"] = "Expense", (
            "=SUMIFS(Transactions!$F:$F,Transactions!$B:$B,$B$1,Transactions!$C:$C,$B$2)"
        )
        # Monthly trend block (account-filtered, every month).
        ws_calc["A6"], ws_calc["B6"], ws_calc["C6"], ws_calc["D6"] = (
            "Month", "Income", "Expense", "Net",
        )
        for j, month in enumerate(months):
            r = 7 + j
            ws_calc.cell(row=r, column=1, value=month)
            ws_calc.cell(
                row=r, column=2,
                value=f"=SUMIFS(Transactions!$G:$G,Transactions!$B:$B,$A{r},Transactions!$C:$C,$B$2)",
            )
            ws_calc.cell(
                row=r, column=3,
                value=f"=SUMIFS(Transactions!$F:$F,Transactions!$B:$B,$A{r},Transactions!$C:$C,$B$2)",
            )
            ws_calc.cell(row=r, column=4, value=f"=B{r}-C{r}")
        last_month_row = 6 + len(months)
        ws_calc.sheet_state = "hidden"

        # --- dashboard --------------------------------------------------------
        ws = book.create_sheet("Dashboard")
        ws.sheet_view.showGridLines = False
        ws.sheet_properties.tabColor = NAVY
        ws.column_dimensions["A"].width = 3
        for letter in "BCDEFGHIJKLMN":
            ws.column_dimensions[letter].width = 15

        ws.merge_cells("B2:H2")
        title = ws["B2"]
        title.value = "Financial Dashboard"
        title.font = Font(bold=True, size=20, color=NAVY)
        title.alignment = Alignment(vertical="center")

        # Filters
        for cell, text in (("B4", "Period:"), ("B5", "Account:")):
            ws[cell] = text
            ws[cell].font = Font(bold=True)
        for cell, default in (("C4", "All"), ("C5", "All")):
            c = ws[cell]
            c.value = default
            c.fill = PatternFill("solid", fgColor="FFF2CC")
            c.font = Font(bold=True)
            c.alignment = Alignment(horizontal="center")
            c.border = _BORDER

        dv_period = DataValidation(
            type="list", formula1=f"=Lists!$A$1:$A${len(periods)}", allow_blank=False
        )
        dv_account = DataValidation(
            type="list", formula1=f"=Lists!$C$1:$C${len(accounts)}", allow_blank=False
        )
        ws.add_data_validation(dv_period)
        ws.add_data_validation(dv_account)
        dv_period.add(ws["C4"])
        dv_account.add(ws["C5"])

        # KPI cards
        income = "=SUMIFS(Transactions!$G:$G,Transactions!$B:$B,Calc!$B$1,Transactions!$C:$C,Calc!$B$2)"
        expense = "=SUMIFS(Transactions!$F:$F,Transactions!$B:$B,Calc!$B$1,Transactions!$C:$C,Calc!$B$2)"
        count = "=COUNTIFS(Transactions!$B:$B,Calc!$B$1,Transactions!$C:$C,Calc!$B$2)"
        _kpi_card(ws, "B7", "B8", "Total Income", income, color=GREEN)
        _kpi_card(ws, "D7", "D8", "Total Expense", expense, color=RED)
        _kpi_card(ws, "F7", "F8", "Net", "=B8-D8", color=NAVY)
        _kpi_card(ws, "H7", "H8", "Transactions", count, fmt=_INT_FMT, color=BLUE)

        # Pie: income vs expense (selection)
        pie = PieChart()
        pie.title = "Income vs Expense (selection)"
        pie.add_data(Reference(ws_calc, min_col=5, min_row=2, max_row=3), titles_from_data=False)
        pie.set_categories(Reference(ws_calc, min_col=4, min_row=2, max_row=3))
        pie.dataLabels = DataLabelList()
        pie.dataLabels.showPercent = True
        pie.height, pie.width = 7.5, 11
        ws.add_chart(pie, "B11")

        # Bar: monthly net (account-filtered)
        bar = BarChart()
        bar.type = "col"
        bar.title = "Monthly Net (selected account)"
        bar.add_data(
            Reference(ws_calc, min_col=4, min_row=6, max_row=last_month_row), titles_from_data=True
        )
        bar.set_categories(Reference(ws_calc, min_col=1, min_row=7, max_row=last_month_row))
        bar.legend = None
        bar.height, bar.width = 7.5, 16
        ws.add_chart(bar, "F11")

        # Line: monthly income vs expense
        line = LineChart()
        line.title = "Monthly Income vs Expense"
        line.add_data(
            Reference(ws_calc, min_col=2, max_col=3, min_row=6, max_row=last_month_row),
            titles_from_data=True,
        )
        line.set_categories(Reference(ws_calc, min_col=1, min_row=7, max_row=last_month_row))
        line.height, line.width = 8, 27
        ws.add_chart(line, "B27")

        # Filtered transaction list — universal INDEX/MATCH (works in every Excel).
        # Helper columns on Transactions: L flags rows matching the dashboard
        # selection, M numbers each match 1..k so INDEX/MATCH can pull the k-th row.
        ws_tx = writer.sheets["Transactions"]
        ws_tx["L1"], ws_tx["M1"] = "_match", "_idx"
        for r in range(2, last + 1):
            ws_tx.cell(
                row=r, column=12,
                value=(
                    '=IF(AND('
                    f'OR(Dashboard!$C$4="All",LEFT($B{r},LEN(Dashboard!$C$4))=Dashboard!$C$4),'
                    f'OR(Dashboard!$C$5="All",$C{r}=Dashboard!$C$5)),1,0)'
                ),
            )
            ws_tx.cell(row=r, column=13, value=f'=IF(L{r}=1,SUM($L$2:L{r}),"")')
        ws_tx.column_dimensions["L"].hidden = True
        ws_tx.column_dimensions["M"].hidden = True

        ws["B44"] = "Transactions for current selection"
        ws["B44"].font = Font(bold=True, size=12, color=NAVY)
        ws["B44"].alignment = Alignment(vertical="center")
        headers = list(tx_out.columns)
        money_idx = {headers.index(c) for c in ("debit", "credit", "amount", "balance")}
        for i, h in enumerate(headers):
            c = ws.cell(row=45, column=2 + i, value=h)
            c.font = Font(bold=True, color=WHITE)
            c.fill = PatternFill("solid", fgColor=BLUE)
            c.alignment = Alignment(horizontal="center")
            c.border = _BORDER
        detail_start = 46
        for k in range(n):
            r = detail_start + k
            for i in range(len(headers)):
                src = get_column_letter(i + 1)  # Transactions column A..J
                cell = ws.cell(
                    row=r, column=2 + i,
                    value=(
                        f'=IFERROR(INDEX(Transactions!${src}$2:${src}${last},'
                        f'MATCH({k + 1},Transactions!$M$2:$M${last},0)),"")'
                    ),
                )
                if i == 0:
                    cell.number_format = _DATE_FMT
                elif i in money_idx:
                    cell.number_format = _MONEY_FMT

        # Sheet order + active
        order = ["Dashboard", "Transactions", "Monthly Summary", "Reconciliation", "Calc", "Lists"]
        book._sheets.sort(key=lambda s: order.index(s.title))
        book.active = 0

    return path


output_path = build_dashboard_workbook(
    transactions, monthly, report, OUTPUT_DIR / "Bank-Statement-Analysis.xlsx"
)
print(f"Saved dashboard workbook -> {output_path.resolve()}")
